# Synthetic Floor - GPU Blender Renderer (Colab)

Renders progressive construction stages with Blender Cycles on a Colab GPU.

**Four example buildings** (all 22x14 m footprint, 7 construction stages each):

| Config | Building | Final floor | Highlights |
|---|---|---|---|
| `config/scene.yaml` | single room (original) | raw slab | columns, walls, windows |
| `config/scene_office.yaml` | office room | **tile** | foundation, concrete beams, exterior site, window frames |
| `config/scene_loft.yaml` | loft studio | **wood** | dense column grid, 7 windows, exposed beams |
| `config/scene_warehouse.yaml` | warehouse bay | **epoxy** | tall (4.2 m), heavy beams, wide roller door |

**Construction logic is respected:** earthwork+foundation -> slab+columns -> concrete beams+roof -> masonry walls (openings left) -> services+rough plaster -> **doors & windows installed + paint** -> final floor finish + lit ceiling. Doors and windows are *not* present in the early stages.

**Production features:** every output is mirrored to Google Drive (crash-safe), runs **resume** with the same run name, each scene exports a valid **IFC4** file (beams, columns, windows, doors, footings) for the main pipeline, and `--save-blend` writes a downloadable **.blend** you can open in Blender on Windows/macOS.

Set the runtime to **GPU**: *Runtime -> Change runtime type -> T4 (or A100)*.


## 1. Verify GPU, mount Drive, clone the repo, choose an example


In [ ]:
import os, sys, subprocess

RUN_NAME = 'demo'          # re-use the same name to RESUME a previous run
CONFIG   = 'config/scene_office.yaml'   # scene.yaml | scene_office.yaml | scene_loft.yaml | scene_warehouse.yaml

try:
    print('GPU:', subprocess.check_output(['nvidia-smi','-L']).decode().strip())
except Exception:
    print('WARNING: no GPU. Runtime -> Change runtime type -> T4/A100.')

DRIVE_ROOT = None
try:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_ROOT = f'/content/drive/MyDrive/MyCon_Colab/synthetic_floor_7stage/{RUN_NAME}'
    os.makedirs(DRIVE_ROOT, exist_ok=True)
    print('Drive output folder:', DRIVE_ROOT)
except Exception as exc:
    print('Drive not available (not on Colab?):', exc)

if not os.path.isdir('/content/MyCon'):
    subprocess.check_call(['git','clone','--depth','1','https://github.com/sayyedalimrj/MyCon.git','/content/MyCon'])
else:
    subprocess.call(['git','-C','/content/MyCon','pull','--ff-only'])
os.chdir('/content/MyCon/repository')
EXAMPLE = 'examples/synthetic_floor_7stage'
sys.path.insert(0, EXAMPLE + '/src')
print('cwd:', os.getcwd(), '| config:', CONFIG)


## 2. Install Blender 4.2 LTS


In [ ]:
%%bash
set -e
BLENDER_VERSION="4.2.3"
BLENDER_TAR="blender-${BLENDER_VERSION}-linux-x64.tar.xz"
BLENDER_URL="https://download.blender.org/release/Blender4.2/${BLENDER_TAR}"
if [ ! -f "/content/blender/blender" ]; then
  echo "Downloading Blender ${BLENDER_VERSION} ..."
  wget -q "${BLENDER_URL}" -O /tmp/blender.tar.xz
  mkdir -p /content/blender
  tar -xf /tmp/blender.tar.xz -C /content/blender --strip-components=1
  rm /tmp/blender.tar.xz
fi
apt-get -qq install -y libxrender1 libxi6 libxkbcommon0 libsm6 libxxf86vm1 libgl1 libegl1 libxfixes3 > /dev/null 2>&1 || true
/content/blender/blender --version 2>&1 | head -2


## 3. Install Python dependencies


In [ ]:
!pip install -q numpy Pillow PyYAML trimesh imageio imageio-ffmpeg ifcopenshell scipy


## 4. Resolve the example's output folder (for previews / downloads)


In [ ]:
from synthetic_floor.scene_spec import load_scene_spec
from pathlib import Path
SPEC = load_scene_spec(Path(EXAMPLE) / CONFIG)
OUTPUT = SPEC.output.root          # e.g. .../output_office
RENDER_DIR = OUTPUT / 'blender_renders'
print('project   :', SPEC.project_name)
print('output dir:', OUTPUT)
print('stages    :', [s.name for s in SPEC.stages])


## 5. Smoke render - stage 7, debug preset

`--save-blend` writes a downloadable `.blend`; `--mount-drive`/`--drive-root` mirror to Drive; `--resume` skips finished stages; `--strict-render` fails on a blank frame.


In [ ]:
!PYTHONPATH={EXAMPLE}/src \
    python3 {EXAMPLE}/scripts/run_blender_gpu.py \
        --config {EXAMPLE}/{CONFIG} \
        --blender /content/blender/blender \
        --preset debug --stages 7 \
        --mount-drive --drive-root "{DRIVE_ROOT}" \
        --save-blend --resume --strict-render


## 6. Preview a frame + brightness/coverage check


In [ ]:
import numpy as np
from PIL import Image
from IPython.display import display
rgb_dir = RENDER_DIR / 'stage_07' / 'rgb'
pngs = sorted(rgb_dir.glob('frame_*.png'))
print(f'{len(pngs)} frame(s) rendered')
if pngs:
    mid = pngs[len(pngs)//2]
    arr = np.asarray(Image.open(mid).convert('RGB'), dtype=np.float32)
    lum = arr.mean(axis=2); white = float((lum > 250).mean())
    print(f'{mid.name}: mean={lum.mean():.1f} std={lum.std():.2f} white_frac={white:.3f}')
    print('Looks OK' if (white < 0.97 and lum.std() > 3.0) else 'LOOKS BLANK - check render log')
    display(Image.open(mid))


## 7. Full 7-stage render (balanced, resumable, saves .blend per stage)


In [ ]:
!PYTHONPATH={EXAMPLE}/src \
    python3 {EXAMPLE}/scripts/run_blender_gpu.py \
        --config {EXAMPLE}/{CONFIG} \
        --blender /content/blender/blender \
        --preset balanced --frames 120 --device OPTIX \
        --mount-drive --drive-root "{DRIVE_ROOT}" \
        --save-blend --resume


## 8. Resume status (portable run-state manifest)


In [ ]:
import json
rs = OUTPUT / 'manifests' / 'run_state_blender_gpu.json'
if rs.exists():
    body = json.loads(rs.read_text())
    print('run_id:', body.get('run_id'), '| in_progress:', body.get('in_progress'))
    for sid, st in sorted(body.get('stages', {}).items()):
        dm = st.get('done_marker') or {}
        print(f"  stage {sid}: complete={st['complete']} frames={dm.get('rgb_frame_count')} blend={bool(dm.get('blend_zip'))}")
else:
    print('No run-state yet - run a render first.')


## 9. Inspect the IFC (for the main MyCon pipeline) + camera path

Each stage exports a valid IFC4 BIM file with stable GlobalIds. Feed any stage's IFC into the main pipeline's BIM stages, or diff GUIDs across stages for progress.


In [ ]:
import json
ifc = OUTPUT / 'bim' / 'stage_07.ifc'
print('IFC:', ifc, '|', ifc.stat().st_size if ifc.exists() else 0, 'bytes')
try:
    import ifcopenshell
    m = ifcopenshell.open(str(ifc))
    for cls in ('IfcFooting','IfcColumn','IfcBeam','IfcWall','IfcWindow','IfcDoor','IfcCovering'):
        print(f'  {cls:12s}: {len(m.by_type(cls))}')
except Exception as exc:
    print('ifcopenshell not available:', exc)


## 10. Download the rendered videos + the .blend projects

The `.blend` zips under `output*/blend/` open directly in Blender on Windows/macOS (procedural materials, no external textures).


In [ ]:
from IPython.display import FileLink, display
print('Videos:')
for mp4 in sorted((OUTPUT / 'video').glob('*.mp4')):
    print(f'  {mp4.name} ({mp4.stat().st_size/1024/1024:.1f} MB)'); display(FileLink(str(mp4)))
print('\nBlender projects (.blend zips):')
for z in sorted((OUTPUT / 'blend').glob('*.zip')):
    print(f'  {z.name} ({z.stat().st_size/1024/1024:.1f} MB)'); display(FileLink(str(z)))


## 11. Resilience & resume

- Outputs mirror to `MyDrive/MyCon_Colab/synthetic_floor_7stage/<RUN_NAME>/output/` after every stage; a crash costs at most the current stage.
- **Resume:** re-run cells 1-7 with the same `RUN_NAME` (and `CONFIG`). Finished stages are pulled from Drive and skipped.
- **Another device / Drive account:** copy that run folder over, set the same `RUN_NAME`, and run.
- Switch `CONFIG` to render a different example building.
